# Notebook 01 — Bootstrap Smoke Test

**Phase:** 0 (Plumbing)  
**Purpose:** Validate that the Kaggle ↔ GitHub loop works end-to-end before we stake real training on it.  
**Expected runtime:** ~2 minutes  
**GPU required:** No (CPU is fine — save your GPU quota for Notebook 03)

## What this notebook does

1. Clones the `nhai` repo into `/kaggle/working/`.
2. Installs the ML pipeline dependencies into the Kaggle Python env.
3. Runs `generate_dummies.py` to confirm TensorFlow can convert Keras models → `.tflite` on Kaggle.
4. Loads the three resulting `.tflite` files through `tf.lite.Interpreter` and verifies every shape matches `shared_contracts/`.

## Why it exists

Every later notebook in this folder follows the same scaffold (clone → install → run module → save artifacts). If any of those steps is broken on Kaggle, we want to discover it now, not three days into a fine-tune run.

## What to send back to Claude after running

Paste the **full text output of cells 3, 4, 5, and 6** into chat. Don't summarize. If anything fails, paste the full traceback.

## 1 — Clone the repo

Kaggle wipes `/kaggle/working/` between sessions, so we re-clone every time. Shallow clone keeps it fast.

In [ ]:
import os, subprocess, sys

WORK_DIR = "/kaggle/working"
REPO_URL = "https://github.com/MalayM09/nhai.git"
REPO_DIR = os.path.join(WORK_DIR, "nhai")

if os.path.isdir(REPO_DIR):
    print(f"{REPO_DIR} exists — pulling latest")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print("\nCWD:", os.getcwd())
print("\nMost recent commits:")
print(subprocess.run(["git", "log", "--oneline", "-5"],
                    capture_output=True, text=True).stdout)

## 2 — Install dependencies

Kaggle already has TensorFlow and numpy pre-installed, but we re-pin from `ml_pipeline/requirements.txt` so the env matches the local venv. The `-q` flag keeps the output tidy.

In [ ]:
!pip install -q -r ml_pipeline/requirements.txt

## 3 — Environment sanity check

If TensorFlow's major version drifts from the local venv (TF 2.21), `.tflite` outputs may differ subtly. Worth noting any version delta now.

In [ ]:
import tensorflow as tf, sys, platform

print(f"Python:     {sys.version.split()[0]}")
print(f"Platform:   {platform.platform()}")
print(f"TensorFlow: {tf.__version__}")
print(f"GPUs:       {tf.config.list_physical_devices('GPU') or 'none (CPU runtime)'}")
print(f"Local venv expects TF >= 2.21")

## 4 — Run the dummy `.tflite` generator

Same script that ran locally during the initial repo setup. If TF on Kaggle can produce these three files, every future training notebook can rely on the same `TFLiteConverter` path.

In [ ]:
!python generate_dummies.py

## 5 — Validate every shape against `shared_contracts/`

Loads each dummy `.tflite` through the TFLite interpreter, asserts input/output shapes match the frozen contract. If any model mismatches, the assertion fails — fix `generate_dummies.py` or the contract before proceeding to the next notebook.

In [ ]:
import os, tensorflow as tf

EXPECTED = {
    "blazeface_dummy.tflite":     {"in": [[1, 128, 128, 3]], "out": [[1, 896, 1], [1, 896, 16]]},
    "shufflenet_dummy.tflite":    {"in": [[1, 112, 112, 3]], "out": [[1, 2]]},
    "mobilefacenet_dummy.tflite": {"in": [[1, 112, 112, 3]], "out": [[1, 512]]},
}

MODELS_DIR = "mobile_app/assets/models"
results = {}

for fn, exp in EXPECTED.items():
    path = os.path.join(MODELS_DIR, fn)
    it = tf.lite.Interpreter(model_path=path)
    it.allocate_tensors()
    ins  = [t["shape"].tolist() for t in it.get_input_details()]
    outs = sorted([t["shape"].tolist() for t in it.get_output_details()])
    exp_outs = sorted(exp["out"])
    ok = (ins == exp["in"]) and (outs == exp_outs)
    size_kb = os.path.getsize(path) / 1024.0
    results[fn] = {"ok": ok, "in": ins, "out": outs, "size_kb": round(size_kb, 1)}
    status = "OK     " if ok else "MISMATCH"
    print(f"[{status}] {fn:30s} in={ins} out={outs} size={size_kb:,.1f} KB")

assert all(r["ok"] for r in results.values()), \
    "Shape mismatch — fix before continuing to Notebook 02"
print("\nAll three dummy .tflite files validated. Bootstrap loop works.")

## What to send back to Claude

Paste into chat:

1. The output of **cell 3** (clone + recent commits)
2. The output of **cell 4** (env sanity — TF version, GPU/CPU)
3. The output of **cell 5** (dummy generator log)
4. The output of **cell 6** (the three `[OK]` lines)

If everything is green, the next message will be Notebook 02 — pulling pretrained MediaPipe BlazeFace + FaceMesh `.tflite` files directly, and converting the insightface ONNX MobileFaceNet to a Keras → `.tflite` model. That replaces all three dummies with real (untrained-on-our-data, but pretrained) weights so the mobile bridge starts producing meaningful outputs.

**Do NOT click "Save Version" yet.** Save Version is for runs whose artifacts we want to preserve. This is a smoke test — its artifacts (the dummy `.tflite` files) are already in the repo. Just close the session when done.